[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_XORViz/NN_DL_ch01_XORViz.ipynb)

# NN_DL_ch01_XORViz

**XOR Problem: MLP Decision Boundary & Hidden Space Visualization**

The XOR problem is not linearly separable. We train a small MLP and visualize how the hidden layer transforms the input space.

In [ ]:
!pip install -q torch matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# XOR Data & MLP
import torch.nn as nn
import torch.optim as optim

X_xor = torch.FloatTensor([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = torch.FloatTensor([[0], [1], [1], [0]])

class XOR_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.output = nn.Linear(4, 1)
    def forward(self, x):
        h = torch.tanh(self.hidden(x))
        return torch.sigmoid(self.output(h))
    def hidden_rep(self, x):
        return torch.tanh(self.hidden(x))

model = XOR_MLP()
optimizer = optim.Adam(model.parameters(), lr=0.05)
criterion = nn.BCELoss()

for epoch in range(2000):
    pred = model(X_xor)
    loss = criterion(pred, y_xor)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

with torch.no_grad():
    preds = model(X_xor)
print("XOR predictions after training:")
for i in range(4):
    print(f"  {X_xor[i].tolist()} -> {preds[i].item():.4f} (target: {y_xor[i].item():.0f})")

In [ ]:
# Decision Boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

model.eval()
with torch.no_grad():
    zz = model(grid).numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 8))
contour = ax.contourf(xx, yy, zz, levels=50, cmap='RdBu_r', alpha=0.8)
ax.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
plt.colorbar(contour, ax=ax, label='P(y=1)')

colors = [MAIN_BLUE if y == 0 else IDA_RED for y in y_xor.ravel()]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=300, edgecolors='black', linewidths=2, zorder=5)
for i, (xi, yi) in enumerate(X_xor):
    ax.annotate(f'({xi:.0f},{yi:.0f})->{y_xor[i].item():.0f}',
                (xi.item(), yi.item()), textcoords="offset points",
                xytext=(15, 10), fontsize=11, fontweight='bold')

ax.set_xlabel('$x_1$', fontsize=14); ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('XOR - MLP Decision Boundary', fontweight='bold', fontsize=15)
save_fig('nn_ch01_xor_mlp_solution')

In [ ]:
# Hidden Space Visualization
with torch.no_grad():
    hidden_xor = model.hidden_rep(X_xor).numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
c_list = [MAIN_BLUE if y == 0 else IDA_RED for y in y_xor.ravel()]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=c_list, s=300, edgecolors='black', linewidths=2, zorder=5)
ax.set_title('Input Space (not separable)', fontweight='bold', fontsize=14)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.annotate('No single line\nseparates classes', xy=(0.5, 0.5), fontsize=12,
            ha='center', va='center', color='grey',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax = axes[1]
for i in range(4):
    color = MAIN_BLUE if y_xor[i].item() == 0 else IDA_RED
    ax.scatter(hidden_xor[i, 0], hidden_xor[i, 1], c=color, s=300,
               edgecolors='black', linewidths=2, zorder=5)
    ax.annotate(f'({X_xor[i,0]:.0f},{X_xor[i,1]:.0f})',
                (hidden_xor[i, 0], hidden_xor[i, 1]),
                textcoords="offset points", xytext=(10, 10), fontsize=11, fontweight='bold')
ax.set_title('Hidden Space (linearly separable!)', fontweight='bold', fontsize=14)
ax.set_xlabel('$h_1$'); ax.set_ylabel('$h_2$')

plt.suptitle('XOR: How the Hidden Layer Transforms Feature Space', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch01_hidden_space')

## Summary

The MLP solves XOR by transforming the input space through its hidden layer into a representation where the classes become **linearly separable**.